In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### 1. LOAD CSV (HEADER DI BARIS KE-2)

In [2]:
df = pd.read_csv(
    "daraset baruu.csv",
    sep=';',
    encoding='latin1',
    header=1
)

df.columns = [
    'dummy',
    'no',
    'tanggal',
    'pendapatan',
    'tambahan_kerja_lembur',
    'uang_keluar',
    'total_pendapatan'
]

df = df[['tanggal', 'total_pendapatan']]
print("Baris awal:", len(df))


Baris awal: 392


In [3]:
df['total_pendapatan'] = (
    df['total_pendapatan']
    .astype(str)
    .str.replace('.', '', regex=False)
)

df['total_pendapatan'] = pd.to_numeric(
    df['total_pendapatan'],
    errors='coerce'
)

print("Setelah bersih angka:", len(df))


Setelah bersih angka: 392


In [4]:
bulan_id = {
    'januari': 'January',
    'februari': 'February',
    'maret': 'March',
    'april': 'April',
    'mei': 'May',
    'juni': 'June',
    'juli': 'July',
    'agustus': 'August',
    'september': 'September',
    'oktober': 'October',
    'november': 'November',
    'desember': 'December'
}

df['tanggal'] = df['tanggal'].astype(str).str.lower()

for indo, eng in bulan_id.items():
    df['tanggal'] = df['tanggal'].str.replace(indo, eng, regex=False)


In [5]:
df['tanggal'] = pd.to_datetime(
    df['tanggal'],
    format='%d %B %Y',
    errors='coerce'
)

print("Tanggal valid:", df['tanggal'].notna().sum())
print("Pendapatan valid:", df['total_pendapatan'].notna().sum())


Tanggal valid: 249
Pendapatan valid: 327


In [6]:
df = df.dropna(subset=['tanggal', 'total_pendapatan'])
print("Baris final:", len(df))
print(df.head())


Baris final: 246
     tanggal  total_pendapatan
1 2025-01-02         1935000.0
2 2025-01-03         1435000.0
3 2025-01-04         1592000.0
4 2025-01-05          990000.0
5 2025-01-06          890000.0


In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))
df['scaled'] = scaler.fit_transform(df[['total_pendapatan']])

print(df[['total_pendapatan', 'scaled']].head())


   total_pendapatan    scaled
1         1935000.0  0.597682
2         1435000.0  0.432119
3         1592000.0  0.484106
4          990000.0  0.284768
5          890000.0  0.251656


In [8]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:i+time_step])
        y.append(data[i+time_step])
    return np.array(X), np.array(y)

time_step = 3
data = df['scaled'].values

X, y = create_dataset(data, time_step)

# reshape ke format LSTM [samples, time_step, features]
X = X.reshape(X.shape[0], X.shape[1], 1)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (243, 3, 1)
Shape y: (243,)


In [9]:
model = Sequential()
model.add(LSTM(1, input_shape=(time_step, 1)))
model.add(Dense(1))

model.compile(
    optimizer='adam',
    loss='mse'
)

model.summary()

C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             2 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14 (56.00 B)

 Trainable params: 14 (56.00 B)

 Non-trainable params: 0 (0.00 B)

### Split Data

In [10]:
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (194, 3, 1)
Test : (49, 3, 1)


In [11]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)


Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0717 - val_loss: 0.0343
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0648 - val_loss: 0.0294
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0584 - val_loss: 0.0252
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0529 - val_loss: 0.0215
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0480 - val_loss: 0.0183
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0437 - val_loss: 0.0155
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0399 - val_loss: 0.0134
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0369 - val_loss: 0.0116
Epoch 9/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0343 - val_loss: 0.0101
Epoch 10/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0319 - val_loss: 0.0089
Epoch 11/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0299 - val_loss: 0.0079
Epoch 12/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0281 - val

Proses pelatihan dilakukan menggunakan optimizer Adam dengan fungsi loss Mean Squared Error (MSE) selama 100 epoch.

In [12]:
# PREDIKSI DATA TEST
y_pred = model.predict(X_test)
print("y_pred shape:", y_pred.shape)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
y_pred shape: (49, 1)


In [13]:
y_test_inv = scaler.inverse_transform(y_test.reshape(-1,1))
y_pred_inv = scaler.inverse_transform(y_pred)

print("Contoh aktual:", y_test_inv[:5].flatten())
print("Contoh prediksi:", y_pred_inv[:5].flatten())


Contoh aktual: [580000. 580000. 480000. 900000. 470000.]
Contoh prediksi: [793453.3  794815.25 795397.9  795629.9  794060.75]


In [14]:
mae = mean_absolute_error(y_test_inv, y_pred_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))

print("MAE (Rp) :", mae)
print("RMSE (Rp):", rmse)

MAE (Rp) : 213609.5280612245
RMSE (Rp): 250962.0245361805


In [15]:
eval_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE"],
    "Value (Rp)": [mae, rmse]
})

eval_df.to_excel("evaluasi_lstm.xlsx", index=False)
print("✅ Evaluasi disimpan ke evaluasi_lstm.xlsx")


✅ Evaluasi disimpan ke evaluasi_lstm.xlsx


In [16]:
weights = model.layers[0].get_weights()

W = weights[0]   # kernel
U = weights[1]   # recurrent kernel
b = weights[2]   # bias

print("Kernel shape:", W.shape)
print("Recurrent shape:", U.shape)
print("Bias shape:", b.shape)

Kernel shape: (1, 4)
Recurrent shape: (1, 4)
Bias shape: (4,)


In [17]:
# Pisahkan gate
Wf, Wi, Wc, Wo = np.split(W, 4, axis=1)
Uf, Ui, Uc, Uo = np.split(U, 4, axis=1)
bf, bi, bc, bo = np.split(b, 4)

param_df = pd.DataFrame({
    "Parameter": [
        "Wf","Uf","bf",
        "Wi","Ui","bi",
        "Wc","Uc","bc",
        "Wo","Uo","bo"
    ],
    "Nilai": [
        Wf.mean(), Uf.mean(), bf.mean(),
        Wi.mean(), Ui.mean(), bi.mean(),
        Wc.mean(), Uc.mean(), bc.mean(),
        Wo.mean(), Uo.mean(), bo.mean()
    ]
})

param_df.to_excel("parameter_lstm.xlsx", index=False)
print("✅ Bobot LSTM disimpan ke parameter_lstm.xlsx")


✅ Bobot LSTM disimpan ke parameter_lstm.xlsx
